## **Model Evaluation**

In [ ]:
import json
import re
import time
import os

import pandas as pd
import requests
from tqdm import tqdm

df = pd.read_csv("dataset.csv")

MODELS = [
  "meta-llama/Llama-3.1-8B-Instruct",
  "deepseek-ai/DeepSeek-V3.1-Terminus"
]

NUM_ITERATIONS = 4

# Never hard-code your token. In Google Colab add it under "Secrets" (key icon)
# with the name HF_API_TOKEN; locally, export it as an environment variable.
try:
    from google.colab import userdata
    HF_API_TOKEN = userdata.get("HF_API_TOKEN")
except Exception:
    HF_API_TOKEN = os.environ.get("HF_API_TOKEN")

if not HF_API_TOKEN:
    raise ValueError(
        "HF_API_TOKEN is not set. Set it as an environment variable."
    )

PROMPT_TEMPLATES = {
    "baseline": """Classify the sentiment of the following text as positive, negative, or neutral.
Text: {text}
Sentiment:""",

    "instruction": """You are given a short piece of text. Read the full text carefully, including
tone and any implied meaning such as sarcasm, before deciding. Classify its
overall sentiment as exactly one of: positive, negative, or neutral.
Respond with only the single label word.
Text: {text}
Sentiment:""",

    "role": """You are an experienced sentiment analysis expert who specialises in detecting
nuance, sarcasm, and subtle tone in customer feedback. Analyse the sentiment
of the text below as positive, negative, or neutral.
Text: {text}
Sentiment:""",

    "structured": """Classify the sentiment of the text below. Respond only in the following JSON
format with no other text: {{"sentiment": "positive"}}
Text: {text}""",
}



def query_huggingface(model_name, prompt, max_retries=3, initial_delay=2):

    url = "https://router.huggingface.co/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {HF_API_TOKEN}",
        "Content-Type": "application/json",
    }

    payload = {
        "model": model_name,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "max_tokens": 300,
        "temperature": 0.1,
    }

    delay = initial_delay

    for attempt in range(max_retries):
        try:
            response = requests.post(
                url,
                headers=headers,
                json=payload,
                timeout=60
            )

            print("HTTP STATUS:", response.status_code)

            if response.status_code == 200:
                data = response.json()
                return data["choices"][0]["message"]["content"]

            elif response.status_code == 429:
                print(f"Rate limited. Retry {attempt + 1}/{max_retries}")
                time.sleep(delay)
                delay *= 2

            elif response.status_code in [401, 403, 404, 422]:
                print("API request failed:", response.text)
                break

            elif response.status_code >= 500:
                print("Server error. Retrying...")
                time.sleep(delay)
                delay *= 2

            else:
                print("API request failed:", response.text)
                break

        except requests.RequestException as e:
            print("REQUEST ERROR:", e)
            time.sleep(delay)
            delay *= 2

    return ""



def parse_output(raw_output, category):

    raw_output = str(raw_output).strip()

    if not raw_output:
        return "unknown"

    if category == "structured":
        try:
            match = re.search(r"\{.*?\}", raw_output, re.DOTALL)

            if match:
                data = json.loads(match.group(0))

                sentiment = str(
                    data.get("sentiment", "")
                ).lower().strip()

                if sentiment in ["positive", "negative", "neutral"]:
                    return sentiment

        except (json.JSONDecodeError, AttributeError, TypeError):
            pass

    raw_lower = raw_output.lower().strip()

    if raw_lower in ["positive", "negative", "neutral"]:
        return raw_lower

    match = re.search(
        r"\b(positive|negative|neutral)\b",
        raw_lower
    )

    if match:
        return match.group(1)

    return "unknown"



results = []

total_models = len(MODELS)
total_samples = len(df)
total_categories = len(PROMPT_TEMPLATES)

total_requests = (
    total_models
    * NUM_ITERATIONS
    * total_samples
    * total_categories
)

print("\n" + "=" * 70)
print("STARTING MODEL EVALUATION")
print("=" * 70)
print(f"Models: {total_models}")
print(f"Dataset samples: {total_samples}")
print(f"Prompt categories: {total_categories}")
print(f"Iterations per model: {NUM_ITERATIONS}")
print(f"Total planned API requests: {total_requests}")
print("=" * 70)


for model_index, model_name in enumerate(MODELS, start=1):

    print("\n" + "#" * 70)
    print(f"MODEL {model_index}/{total_models}: {model_name}")
    print("#" * 70)

    for iteration in range(1, NUM_ITERATIONS + 1):

        print("\n" + "-" * 70)
        print(
            f"MODEL: {model_name}\n"
            f"ITERATION: {iteration}/{NUM_ITERATIONS}"
        )
        print("-" * 70)

        for _, row in tqdm(
            df.iterrows(),
            total=len(df),
            desc=f"{model_name} | Iteration {iteration}/{NUM_ITERATIONS}"
        ):

            text_input = str(row["text"])

            record = {
                "id": row["id"],
                "text": text_input,
                "model": model_name,
                "iteration": iteration,
                "expected_label": str(
                    row["expected_label"]
                ).strip().lower(),
            }

            for category, template in PROMPT_TEMPLATES.items():

                print(
                    f"\nModel: {model_name}\n"
                    f"Iteration: {iteration}/{NUM_ITERATIONS}\n"
                    f"Sample ID: {row['id']}\n"
                    f"Prompt category: {category}"
                )

                prompt = template.format(text=text_input)

                raw_res = query_huggingface(
                    model_name,
                    prompt
                )

                parsed_res = parse_output(
                    raw_res,
                    category
                )

                print("Parsed prediction:", parsed_res)

                record[f"{category}_output"] = parsed_res

                record[f"{category}_raw_output"] = raw_res

                time.sleep(0.2)

            results.append(record)

        print(
            f"\nCompleted iteration {iteration}/{NUM_ITERATIONS} "
            f"for model: {model_name}"
        )



outputs_df = pd.DataFrame(results)

outputs_df.to_csv(
    "model_outputs.csv",
    index=False,
    encoding="utf-8"
)

print("\n" + "=" * 70)
print("EVALUATION COMPLETED")
print("=" * 70)
print(f"Total result rows: {len(outputs_df)}")
print("Results saved to: model_outputs.csv")
print("=" * 70)

## **Calculating Model Accuracy**

In [ ]:
import csv
from collections import defaultdict

categories = ["baseline", "instruction", "role", "structured"]


models_metrics = defaultdict(
    lambda: defaultdict(
        lambda: {
            c: {
                "correct": 0,
                "total": 0,
                "errors": []
            }
            for c in categories
        }
    )
)


with open("model_outputs.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)

    for row in reader:

        model = row.get("model", "unknown_model").strip()
        iteration = row.get("iteration", "unknown").strip()
        expected = row["expected_label"].strip().lower()

        for c in categories:

            output = row[f"{c}_output"].strip().lower()

            models_metrics[model][iteration][c]["total"] += 1

            if output == expected:
                models_metrics[model][iteration][c]["correct"] += 1
            else:
                models_metrics[model][iteration][c]["errors"].append(
                    row["id"]
                )


for model, iterations in models_metrics.items():

    print("\n" + "=" * 70)
    print(f"MODEL: {model}")
    print("=" * 70)

    for iteration, counts in iterations.items():

        print(f"\nIteration: {iteration}")
        print("-" * 60)

        print(
            f"{'Prompt Category':<17}"
            f"{'Correct':<10}"
            f"{'Total':<10}"
            f"{'Accuracy':<10}"
        )

        print("-" * 47)

        for c in categories:

            correct = counts[c]["correct"]
            total = counts[c]["total"]

            acc = (
                correct / total * 100
                if total > 0
                else 0.0
            )

            print(
                f"{c:<17}"
                f"{correct:<10}"
                f"{total:<10}"
                f"{acc:.2f}%"
            )

        print("\nMisclassified sample IDs:")

        for c in categories:

            err_list = (
                ", ".join(map(str, counts[c]["errors"]))
                if counts[c]["errors"]
                else "none"
            )

            print(f"  {c:<15}: {err_list}")


all_metrics_rows = []

fieldnames = [
    "model",
    "iteration",
    "prompt_category",
    "correct",
    "total",
    "accuracy_percent",
    "misclassified_ids",
]

for model, iterations in models_metrics.items():

    for iteration, counts in iterations.items():

        for c in categories:

            correct = counts[c]["correct"]
            total = counts[c]["total"]

            acc = (
                round(correct / total * 100, 2)
                if total > 0
                else 0.0
            )

            err_ids = (
                ";".join(map(str, counts[c]["errors"]))
                if counts[c]["errors"]
                else "none"
            )

            row_data = {
                "model": model,
                "iteration": iteration,
                "prompt_category": c,
                "correct": correct,
                "total": total,
                "accuracy_percent": acc,
                "misclassified_ids": err_ids,
            }

            all_metrics_rows.append(row_data)


with open(
    "model_metrics_all.csv",
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=fieldnames
    )

    writer.writeheader()
    writer.writerows(all_metrics_rows)

print("\nSaved combined summary file: model_metrics_all.csv")

Average Accuracy of prompts

In [ ]:
import pandas as pd

df = pd.read_csv('model_metrics_all.csv')

overall_avg = df.groupby('prompt_category')['accuracy_percent'].mean().reset_index()
overall_avg.columns = ['Prompt Technique', 'Average Accuracy (%)']

overall_avg.to_csv('overall_average_accuracy.csv', index=False)


print("Saved 'overall_average_accuracy.csv'' successfully.")

## **Calculating Reliability and Consistency Metrics**

In [ ]:
import csv
from collections import defaultdict
from itertools import combinations


INPUT_FILE = "model_outputs.csv"
OUTPUT_FILE = "reliability_consistency_all.csv"

categories = [
    "baseline",
    "instruction",
    "role",
    "structured"
]


data = defaultdict(
    lambda: defaultdict(
        lambda: defaultdict(dict)
    )
)

with open(INPUT_FILE, newline="", encoding="utf-8") as f:

    reader = csv.DictReader(f)

    for row in reader:

        model = row["model"].strip()
        iteration = row["iteration"].strip()
        sample_id = row["id"].strip()

        for category in categories:

            prediction = row[
                f"{category}_output"
            ].strip().lower()

            data[model][category][sample_id][iteration] = prediction



def cohens_kappa(labels1, labels2):

    if not labels1:
        return 0.0

    total = len(labels1)

    observed = sum(
        a == b
        for a, b in zip(labels1, labels2)
    ) / total

    labels = set(labels1) | set(labels2)

    expected = 0.0

    for label in labels:

        p1 = labels1.count(label) / total
        p2 = labels2.count(label) / total

        expected += p1 * p2

    if expected == 1:
        return 1.0

    return (
        (observed - expected)
        / (1 - expected)
    )



results = []


for model, categories_data in data.items():

    for category, samples in categories_data.items():


        iteration_set = set()

        for sample_data in samples.values():

            iteration_set.update(
                sample_data.keys()
            )

        iterations = sorted(
            iteration_set,
            key=lambda x: int(x)
            if x.isdigit()
            else x
        )

        num_iterations = len(iterations)

        if num_iterations < 2:

            print(
                f"Skipping {model} | {category} "
                f"- only {num_iterations} iteration"
            )

            continue


        complete_samples = []

        for sample_id, sample_data in samples.items():

            if all(
                iteration in sample_data
                for iteration in iterations
            ):

                complete_samples.append(
                    sample_id
                )

        total_samples = len(
            complete_samples
        )

        if total_samples == 0:
            continue


        sample_reliabilities = []

        for sample_id in complete_samples:

            predictions = [
                samples[sample_id][iteration]
                for iteration in iterations
            ]

            prediction_counts = defaultdict(int)

            for prediction in predictions:
                prediction_counts[prediction] += 1

            max_frequency = max(
                prediction_counts.values()
            )

            sample_reliability = (
                max_frequency /
                num_iterations
            )

            sample_reliabilities.append(
                sample_reliability
            )

        reliability_percent = (
            sum(sample_reliabilities)
            / len(sample_reliabilities)
            * 100
        )


        fully_consistent = 0

        for sample_id in complete_samples:

            predictions = [
                samples[sample_id][iteration]
                for iteration in iterations
            ]

            if len(set(predictions)) == 1:

                fully_consistent += 1

        consistency_percent = (
            fully_consistent
            / total_samples
            * 100
        )


        pairwise_agreements = []
        pairwise_kappas = []

        for iter1, iter2 in combinations(
            iterations,
            2
        ):

            labels1 = []
            labels2 = []

            for sample_id in complete_samples:

                labels1.append(
                    samples[sample_id][iter1]
                )

                labels2.append(
                    samples[sample_id][iter2]
                )


            agreement = sum(
                a == b
                for a, b in zip(
                    labels1,
                    labels2
                )
            ) / len(labels1)

            pairwise_agreements.append(
                agreement * 100
            )


            kappa = cohens_kappa(
                labels1,
                labels2
            )

            pairwise_kappas.append(
                kappa
            )

        average_pairwise_agreement = (
            sum(pairwise_agreements)
            / len(pairwise_agreements)
        )

        average_kappa = (
            sum(pairwise_kappas)
            / len(pairwise_kappas)
        )


        result = {

            "model": model,

            "prompt_category": category,

            "num_iterations": num_iterations,

            "total_samples": total_samples,

            "reliability_percent": round(
                reliability_percent,
                2
            ),

            "consistency_percent": round(
                consistency_percent,
                2
            ),

            "average_pairwise_agreement_percent": round(
                average_pairwise_agreement,
                2
            ),

            "average_cohens_kappa": round(
                average_kappa,
                4
            ),
        }

        results.append(result)



fieldnames = [
    "model",
    "prompt_category",
    "num_iterations",
    "total_samples",
    "reliability_percent",
    "consistency_percent",
    "average_pairwise_agreement_percent",
    "average_cohens_kappa",
]


with open(
    OUTPUT_FILE,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=fieldnames
    )

    writer.writeheader()
    writer.writerows(results)



print("\n" + "=" * 90)
print("RELIABILITY AND CONSISTENCY EVALUATION")
print("=" * 90)

for row in results:

    print("\n" + "-" * 90)

    print(
        f"Model: {row['model']}"
    )

    print(
        f"Prompt Category: "
        f"{row['prompt_category']}"
    )

    print(
        f"Iterations: "
        f"{row['num_iterations']}"
    )

    print(
        f"Samples: "
        f"{row['total_samples']}"
    )

    print(
        f"Reliability: "
        f"{row['reliability_percent']}%"
    )

    print(
        f"Consistency: "
        f"{row['consistency_percent']}%"
    )

    print(
        f"Average Pairwise Agreement: "
        f"{row['average_pairwise_agreement_percent']}%"
    )

    print(
        f"Cohen's Kappa: "
        f"{row['average_cohens_kappa']}"
    )


print("\n" + "=" * 90)
print(
    f"Saved results to: {OUTPUT_FILE}"
)
print("=" * 90)

Averaging Reliability and consistency across prompts


In [ ]:
import pandas as pd

df_rel = pd.read_csv('reliability_consistency_all.csv')

metrics = [
    'reliability_percent',
    'consistency_percent',
    'average_pairwise_agreement_percent',
    'average_cohens_kappa'
]

overall_rel_avg = df_rel.groupby('prompt_category')[metrics].mean().reset_index()
overall_rel_avg.columns = [
    'Prompt Category',
    'Reliability (%)',
    'Consistency (%)',
    'Pairwise Agreement (%)',
    'Cohen\'s Kappa'
]

overall_rel_avg.to_csv('overall_average_reliability.csv', index=False)

Calculating Correlations

In [ ]:
import pandas as pd

metrics_df = pd.read_csv('model_metrics_all.csv')
reliability_df = pd.read_csv('reliability_consistency_all.csv')

acc_summary = metrics_df.groupby(['model', 'prompt_category'])['accuracy_percent'].mean().reset_index()

df = pd.merge(acc_summary, reliability_df, on=['model', 'prompt_category'])

def get_strength_label(r):
    abs_r = abs(r)
    if abs_r >= 0.80:
        strength = "Very strong"
    elif abs_r >= 0.60:
        strength = "Strong"
    elif abs_r >= 0.40:
        strength = "Moderate"
    elif abs_r >= 0.20:
        strength = "Weak"
    else:
        strength = "Very weak / Negligible"

    direction = "positive" if r >= 0 else "negative"
    return f"{strength} {direction}"

target_metrics = [
    ('reliability_percent', 'Accuracy vs Reliability (%)'),
    ('consistency_percent', 'Accuracy vs Consistency (%)'),
    ('average_pairwise_agreement_percent', 'Accuracy vs Pairwise Agreement (%)'),
    ('average_cohens_kappa', "Accuracy vs Cohen's Kappa")
]

results = []
for col, label in target_metrics:
    r = df['accuracy_percent'].corr(df[col])
    results.append({
        "Metric Pair": label,
        "Correlation Coefficient (r)": round(r, 4),
        "Strength": get_strength_label(r)
    })

correlations_df = pd.DataFrame(results)

correlations_df.to_csv('accuracy_reliability_exact_correlations.csv', index=False)

print("=== Metric Pair Correlations ===")
print(correlations_df.to_string(index=False))